In [1]:
import sys
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import os
import gc
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sparse
from scvi.data import synthetic_iid
from scvi.dataloaders import AnnDataLoader
#import diopy
import warnings
warnings.filterwarnings('ignore')
from cell2location import run_colocation
from cell2location.models import Cell2location, RegressionModel
import torch
import cell2location
from matplotlib.lines import Line2D
import umap
from scvi.model import CondSCVI,DestVI
import scvi
import matplotlib as mpl
from anndata import AnnData
import pathlib
import matplotlib as mpl
import skimage
import tangram as tg
sc.logging.print_header()

Global seed set to 0
/opt/conda/lib/python3.8/site-packages/flax/core/frozen_dict.py:169: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(


scanpy==1.9.2 anndata==0.8.0 umap==0.5.3 numpy==1.22.4 scipy==1.9.1 pandas==1.5.1 scikit-learn==1.2.1 statsmodels==0.13.5 python-igraph==0.10.4 pynndescent==0.5.8


In [2]:
T_names = ["T808", "T809", "T906", "T758", "T996", "T756", "T993", "T989"]
scdir = "/data/snRNA_downsample.h5ad"
for profile in T_names:
    # Dynamically Generate Spatial Data File Paths
    stdir = "/data/" + profile + "_Spatial-id.h5ad"
    
    # read data
    adata_sc = sc.read_h5ad(scdir)
    adata_st = sc.read_h5ad(stdir)

    # set target col
    key = 'subclass.v4'
    
    # Calculate differentially expressed genes for each cell population.
    sc.tl.rank_genes_groups(adata_sc, groupby=key, use_raw=False)
    
    # Retrieve the top 100 genes.
    markers_df = pd.DataFrame(adata_sc.uns["rank_genes_groups"]["names"]).iloc[0:100, :]
    markers = list(np.unique(markers_df.melt().value.values))
    
    # Check whether marker genes are present in the spatial data.
    tg.pp_adatas(adata_sc, adata_st, genes=markers)
    
    # Mapping Single-Cell Data to Spatial Data Using Tangram
    ad_map = tg.map_cells_to_space(adata_sc, adata_st,
                                   mode="clusters",  
                                   cluster_label=key,
                                   density_prior='rna_count_based',
                                   num_epochs=1000,
                                   device="cpu")
    
    # Project cell annotations onto spatial data.
    tg.project_cell_annotations(ad_map, adata_st, annotation=key)
    
    # Get a list of unique annotations.
    annotation_list = list(pd.unique(adata_sc.obs[key]))
    
    # Save the predicted cell type results as a CSV file.
    output_file = f"/data/Tangram_{profile}.cellbin_pret.csv"
    adata_st.obsm['tangram_ct_pred'].sort_index(axis=1).columns
    adata_st.obsm['tangram_ct_pred'].to_csv(output_file)

    print(f"Profile {profile} processed successfully. Results saved to {output_file}")

INFO:root:1571 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:40397 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.
INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 1571 genes and rna_count_based density_prior in clusters mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.183, KL reg: 0.161
Score: 0.279, KL reg: 0.001
Score: 0.282, KL reg: 0.001
Score: 0.282, KL reg: 0.001
Score: 0.283, KL reg: 0.001
Score: 0.283, KL reg: 0.001
Score: 0.283, KL reg: 0.001
Score: 0.283, KL reg: 0.001
Score: 0.283, KL reg: 0.001
Score: 0.283, KL reg: 0.001


INFO:root:Saving results..
INFO:root:spatial prediction dataframe is saved in `obsm` `tangram_ct_pred` of the spatial AnnData.


Profile T808 processed successfully. Results saved to /data/work/Layer division/01_Result/Tangram_T808.cellbin_pret.csv


INFO:root:1571 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:40680 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.
INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 1571 genes and rna_count_based density_prior in clusters mode...
INFO:root:Printing scores every 100 epochs.


...

Profile T993 processed successfully. Results saved to /data/work/Layer division/01_Result/Tangram_T993.cellbin_pret.csv


INFO:root:1571 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:40498 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.
INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 1571 genes and rna_count_based density_prior in clusters mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.235, KL reg: 0.309
Score: 0.402, KL reg: 0.002
Score: 0.405, KL reg: 0.001
Score: 0.406, KL reg: 0.001
Score: 0.407, KL reg: 0.001
Score: 0.407, KL reg: 0.001
Score: 0.407, KL reg: 0.001
Score: 0.407, KL reg: 0.001
Score: 0.407, KL reg: 0.001
Score: 0.407, KL reg: 0.001


INFO:root:Saving results..
INFO:root:spatial prediction dataframe is saved in `obsm` `tangram_ct_pred` of the spatial AnnData.


Profile T989 processed successfully. Results saved to /data/work/Layer division/01_Result/Tangram_T989.cellbin_pret.csv
